# 🩺 Diabetic Retinopathy Detector — One-Click Colab Runner

**What this notebook does:** takes you from an empty Colab session to a *trained*
model that grades retinal photos, end to end. Just set the runtime to GPU and run
every cell top to bottom.

`Runtime → Change runtime type → Hardware accelerator → GPU`, then
`Runtime → Run all`.

The pipeline is: **get code → install → self-test → download data → smoke-test →
train → evaluate → explain (Grad-CAM) → save**. Each step has a short plain-language
note explaining *why* it's there.

> ⚠️ **Not a medical device.** This is an education / portfolio project. It must
> never be used to make real clinical decisions.

## Step 1 — Check you actually got a GPU

Training a deep network on a CPU is painfully slow; on Colab's free GPU a full run
is ~15–40 min. This cell just confirms a GPU is attached. If it says *no GPU*, go to
`Runtime → Change runtime type` and pick **GPU**, then re-run.

In [ ]:
# nvidia-smi lists the attached GPU (errors harmlessly if none is attached yet).
!nvidia-smi || echo 'No GPU visible — set Runtime → Change runtime type → GPU, then re-run.'

## Step 2 — Get the code

Point `REPO_URL` at your GitHub repository (once you've pushed it). The cell clones
the repo and `cd`s into it. If you've already cloned it, it just
reuses the existing copy — safe to re-run.

**Haven't pushed to GitHub yet?** Upload the project folder to Colab manually (Files
panel → upload) or mount Google Drive, then set `REPO_DIR` to that path and skip the
clone.

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = 'https://github.com/RohinDaCoder23/Diabetic-Retinopathy-Application.git'
REPO_DIR = 'diabetic-retinopathy'  # local folder to clone into

if pathlib.Path(REPO_DIR).exists():
    print(f'Repo folder "{REPO_DIR}" already present — reusing it.')
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())
assert pathlib.Path('config.yaml').exists(), 'config.yaml not found — not in repo root?'

# Guard against a STALE clone (a commit from before the src/models fix).
if not pathlib.Path('src/models/__init__.py').exists():
    raise SystemExit(
        'This clone is missing src/models/ — your GitHub repo has an OUTDATED commit.\n'
        'Push the current code from your machine, then re-run:\n'
        '    git push -u origin main --force\n'
        '(or use the self-contained notebook, which needs no GitHub).')
print('Code looks complete (src/models present).')

## Step 3 — Install the dependencies

Everything is pinned in `requirements.txt` (plus `pytest` for the tests). Colab
already has a matching PyTorch, so this is quick. The second line prints the Torch
version and confirms CUDA is available to Python.

In [ ]:
# Install pinned dependencies
!pip install -q -r requirements.txt -r requirements-dev.txt

# Colab sometimes ships an 'albucore' newer than albumentations 1.4.10 expects,
# which breaks the import (preserve_channel_dim). Repair it if that happens.
try:
    import albumentations  # noqa: F401
except Exception as _e:
    print('Repairing albumentations/albucore compatibility:', _e)
    !pip install -q --force-reinstall --no-deps "albucore==0.0.13"
    import importlib; importlib.invalidate_caches()
    import albumentations  # noqa: F401

import torch, albumentations
print('Torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| albumentations', albumentations.__version__)

## Step 4 — Self-test the code *before* touching data

This proves the code runs correctly on its own. `run_offline_checks.py` exercises the
image-processing core with no data needed; `pytest -m "not slow"` runs the fast unit
tests (models build, transforms, splits). If these are green, any later failure is
about data or compute, not broken code.

In [ ]:
!python tests/run_offline_checks.py
!pytest -m 'not slow' -q

## Step 5 — Download the APTOS 2019 dataset from Kaggle

The real fundus images live on Kaggle. You need a free Kaggle account and must
**accept the competition rules** on the
[APTOS 2019 page](https://www.kaggle.com/competitions/aptos2019-blindness-detection/rules)
first, or the download returns a 403.

Then: Kaggle → your avatar → *Settings* → *API* → **Create New API Token**. That
downloads `kaggle.json`. Run the cell below and upload that file when prompted.

This is the single most common place people get stuck — take it slow.

In [ ]:
import pathlib, os
DATA_DIR = 'data/aptos2019'

if pathlib.Path(f'{DATA_DIR}/train.csv').exists():
    print('APTOS already present — skipping download.')
else:
    from google.colab import files
    print('Upload your kaggle.json (Kaggle → Settings → API → Create New API Token):')
    files.upload()  # choose kaggle.json
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.replace('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    !pip install -q kaggle
    os.makedirs(DATA_DIR, exist_ok=True)
    !kaggle competitions download -c aptos2019-blindness-detection -p {DATA_DIR}
    !cd {DATA_DIR} && unzip -q -o aptos2019-blindness-detection.zip && rm -f aptos2019-blindness-detection.zip
    print('Done. Contents of', DATA_DIR, ':')
    !ls {DATA_DIR}

In [ ]:
# Sanity-check the data: how many images, and the class balance?
import pandas as pd, pathlib
df = pd.read_csv('data/aptos2019/train.csv')
n_imgs = len(list(pathlib.Path('data/aptos2019/train_images').glob('*.png')))
print(f'{len(df)} labelled rows | {n_imgs} image files')
print('Class distribution (grade: count):')
print(df['diagnosis'].value_counts().sort_index().to_string())
print('\nNote the imbalance — ~half are grade 0 (No DR). That is WHY we report QWK, '
      'not plain accuracy.')

## Step 6 — Smoke-test training on the real data (~1 minute)

Before a long run, do a 1-epoch / 2-batch dry run. If it finishes and writes a
checkpoint, the whole train→save chain works on this data. Always do this first.

In [ ]:
!python src/train.py --config config.yaml --model resnet50 --smoke-test

## Step 7 — Train for real

Pick a model and a number of epochs. **Recommended default: `resnet50`** — robust and
well understood. `efficientnet_b0` gives the best accuracy-per-size; `custom_cnn` is
the from-scratch baseline you expect to *lose* (train it to prove the point).

Early stopping means it may finish before hitting `EPOCHS`. Fewer epochs = faster but
lower scores; 15–30 is a good range on Colab.

In [ ]:
MODEL  = 'resnet50'   # 'resnet50' | 'efficientnet_b0' | 'densenet121' | 'custom_cnn'
EPOCHS = 20           # early stopping may end sooner

!python src/train.py --config config.yaml --model {MODEL} --epochs {EPOCHS}

## Step 8 — Evaluate on the held-out test set

This runs the trained model on the test split (used only here, once) and computes the
full scorecard. **Lead with QWK and referable-recall**, not accuracy: with ~50% healthy
images a lazy 'always No-DR' model scores ~50% accuracy while catching zero patients.

In [ ]:
!python src/evaluate.py --config config.yaml --model {MODEL}

import json
from IPython.display import Image, display
m = json.load(open(f'reports/metrics_{MODEL}.json'))
print(f"QWK (headline)      : {m['qwk']:.3f}")
print(f"Accuracy            : {m['accuracy']:.3f}")
print(f"Macro F1            : {m['macro_f1']:.3f}")
print(f"Referable recall    : {m['referable']['recall']:.3f}  (grade >= 2 caught)")
cm_png = f'reports/figures/confusion_matrix_{MODEL}.png'
import pathlib
if pathlib.Path(cm_png).exists():
    display(Image(filename=cm_png))

## Step 9 — See *where* the model looked (Grad-CAM)

Grad-CAM paints a heatmap over the regions that drove the prediction. A trustworthy
one sits on lesions, not on the black border or image edge. **Caution:** it shows
correlation, not proof — a confident heatmap can still sit on a wrong answer.

In [ ]:
!python src/gradcam.py --config config.yaml --model {MODEL} --per-grade 2

import glob
from IPython.display import Image, display
panels = sorted(glob.glob(f'reports/figures/gradcam_{MODEL}/*.png'))[:4]
for p in panels:
    display(Image(filename=p))
print(f'{len(panels)} example panels shown (full gallery in reports/figures/gradcam_{MODEL}/).')

## Step 10 — (Optional) Train the full comparison set

Set `RUN_ALL = True` to train every model so you can fill in the README results table
and show the transfer-learning models beating the from-scratch baseline. This takes a
while — leave it running.

In [ ]:
RUN_ALL = False  # flip to True to train + evaluate every model

if RUN_ALL:
    for name in ['custom_cnn', 'resnet50', 'efficientnet_b0', 'densenet121']:
        print(f'\n===== {name} =====')
        !python src/train.py --config config.yaml --model {name} --epochs {EPOCHS}
        !python src/evaluate.py --config config.yaml --model {name}
else:
    print('RUN_ALL is False — skipping the full sweep.')

## Step 11 — Save results to Google Drive (survive a runtime reset)

Colab wipes the filesystem when the runtime recycles. This copies your trained
weights and reports to Drive so you keep them. You can also download the trained
`.pt` to run the Streamlit app locally.

In [ ]:
SAVE_TO_DRIVE = False  # flip to True to copy models/ + reports/ to your Drive

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    dest = '/content/drive/MyDrive/diabetic-retinopathy-results'
    !mkdir -p {dest}
    !cp -r models {dest}/ 2>/dev/null || true
    !cp -r reports {dest}/ 2>/dev/null || true
    print('Copied models/ and reports/ to', dest)
else:
    print('SAVE_TO_DRIVE is False — nothing copied.')

## Step 12 — (Optional) Run the web app from Colab

You can serve the Streamlit app through a tunnel. Uncomment and run. Easier path:
download a trained `*_best.pt` from `models/`, put it in your local repo's `models/`,
and run `streamlit run app/streamlit_app.py` on your own machine.

In [ ]:
# !pip install -q streamlit
# !npm install -g localtunnel  # or use cloudflared
# !streamlit run app/streamlit_app.py &>/content/st.log &
# !npx localtunnel --port 8501
print('Optional — see the commented commands above to expose the app via a tunnel.')

## Recap & next steps

You just: installed the project, self-tested the code, downloaded APTOS, smoke-tested,
trained a model, evaluated it (QWK + confusion matrix), and generated Grad-CAM
explanations.

From here you can: train the full comparison set (Step 10), bump `image.size` in
`config.yaml` to 384/512 for higher accuracy, or open `notebooks/03_results.ipynb` to
fill in the README results table with your real numbers.

> ⚠️ **Reminder:** not a medical device. Research / education only.